# 6 - SHAP compatible

This notebook uses a tree ensemble because tree models are both:

- easy to analyze directly with the built-in `mltsa` tools
- good candidates for external SHAP workflows later on

The notebook itself stays inside `mltsa`. An optional final cell shows how you could hook
the same fitted model into SHAP if that package is installed locally.

In [ ]:
from pathlib import Path
import sys

try:
    import mltsa  # noqa: F401
except ImportError:
    for parent in (Path.cwd(), *Path.cwd().parents):
        src_dir = parent / "src"
        if (src_dir / "mltsa").exists():
            sys.path.insert(0, str(src_dir))
            break
    import mltsa  # noqa: F401

import numpy as np
import matplotlib.pyplot as plt

for parent in (Path.cwd(), *Path.cwd().parents):
    notebooks_dir = parent / "notebooks"
    if notebooks_dir.exists():
        DATA_DIR = notebooks_dir / "_generated"
        break
else:
    DATA_DIR = Path.cwd() / "_generated"

DATA_DIR.mkdir(exist_ok=True, parents=True)
plt.style.use("seaborn-v0_8-whitegrid")
np.set_printoptions(precision=3, suppress=True)

In [ ]:
from sklearn.model_selection import train_test_split

from mltsa.explain import analyze, plot_importances
from mltsa.models import get_model
from mltsa.synthetic import make_1d_dataset

dataset = make_1d_dataset(
    n_trajectories=180,  # Small but stable synthetic dataset
    n_steps=64,  # Total steps per trajectory
    n_features=12,  # Total observed features
    n_relevant=3,  # Hidden ground-truth relevant features
    base_seed=77,  # Reproducible dataset definition
)
window = 28  # Early-time window used for the analysis

X = dataset.X[:, :window, :].reshape(dataset.n_trajectories, -1)
y = dataset.y
feature_names = tuple(
    f"{name}@t{step:03d}"
    for step in range(window)
    for name in dataset.feature_names
)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=0,
    stratify=y,
)

model = get_model(
    "extra_trees",
    n_estimators=240,  # A larger tree ensemble for a stable ranking
    min_samples_leaf=2,  # Mild regularization
)
model.fit(X_train, y_train)
test_accuracy = model.score(X_test, y_test)

explanation = analyze(
    model,
    method="native",
    feature_names=feature_names,
)

base_importance = explanation.importances.reshape(window, dataset.n_features).mean(axis=0)
print(f"Test accuracy: {test_accuracy:.3f}")
print("Ground-truth relevant features:", dataset.relevant_feature_names)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
plot_importances(explanation, top_n=12, ax=axes[0])

colors = [
    "#e76f51" if index in dataset.relevant_features else "#c7ced6"
    for index in range(dataset.n_features)
]
axes[1].bar(np.arange(dataset.n_features), base_importance, color=colors)
axes[1].set_xticks(np.arange(dataset.n_features), dataset.feature_names, rotation=45, ha="right")
axes[1].set_title("Base-feature view")
axes[1].set_ylabel("mean importance")

plt.tight_layout()
plt.show()

We can also persist the explanation result to an HDF5 results file.

In [ ]:
results_path = DATA_DIR / "shap_compatible_results.h5"
explanation_path = explanation.save(results_path, experiment_id="shap_compatible")

print("Results file:", results_path)
print("Saved explanation path:", explanation_path)

## Optional: use the same fitted model with SHAP

This cell is optional on purpose. The tutorial still works without `shap`.

In [ ]:
try:
    import shap
except ImportError:
    print("Optional dependency not installed: pip install shap")
else:
    explainer = shap.TreeExplainer(model.estimator)
    shap_values = explainer.shap_values(X_test[:20])
    summary = np.asarray(shap_values)
    print("SHAP values computed for 20 samples.")
    print("SHAP array shape:", summary.shape)